In [ ]:
import ee
import geemap

ee.Authenticate()
ee.Initialize()


Successfully saved authorization token.


In [ ]:
def get_bands(region, crops, filename):
    """
    Extracts Sentinel-2 time-series data for specific crop types 
    and exports the results as a CSV.
    """

    # ---- 1. Load and Process CDL (Ground Truth) Dataset ----
    # Load the 2021 USDA Cropland Data Layer (CDL) and clip to the study area
    cdl = ee.Image('USDA/NASS/CDL/2021').clip(region)
    cdl_crop = cdl.select('cropland')

    # Calculate pixel frequency (histogram) to see the distribution of crops in the region
    hist = cdl.reduceRegion(
        reducer=ee.Reducer.frequencyHistogram(),
        geometry=region,
        scale=30,
        maxPixels=1e13
    ).getInfo()['cropland']

    total_points = 0
    specified_points = 0

    # Print a summary of how many pixels exist for the user-requested crops
    print("\n--- Crop Distribution (only requested crops) ---")
    for crop_id, _ in crops:
        count = hist.get(str(crop_id), 0)
        print(f"Crop {crop_id}: {count} points")
        specified_points += count

    total_points = sum(hist.values())
    unspecified = total_points - specified_points

    print(f"\nTotal crop points: {total_points}")
    print(f"Unspecified crop points: {unspecified}")

    # ---- 2. Define Sampling Logic ----
    def sample_class(class_id, num_samples):
        """Creates random points within a specific crop type mask."""
        class_mask = cdl_crop.eq(class_id)

        # SelfMask removes pixels that don't match the ID, then sample creates random points
        samples = class_mask.selfMask().sample(
            region=region,
            scale=30,
            numPixels=num_samples,
            geometries=True
        )
        # Attach the crop ID as a property to every point
        return samples.map(lambda f: f.set('crop', class_id))
    

    # ---- 3. Generate the Sample Points Collection ----
    all_samples = ee.FeatureCollection([])

    # Loop through the 'crops' list provided in the function argument to collect points
    for crop_id, num_samples in crops:
        class_samples = sample_class(crop_id, num_samples)
        all_samples = all_samples.merge(class_samples)

    # ---- 4. Prepare Sentinel-2 Imagery ----
    # Define spectral bands of interest (10m and 20m bands)
    bands = ['B2','B3','B4','B5','B6','B7','B8','B8A','B11','B12']

    def mask_s2_clouds(image):
        """Uses the Scene Classification Layer (SCL) to remove clouds and snow."""
        scl = image.select('SCL')
        mask = (
            scl.neq(11)        # Filter out snow
            .And(scl.neq(9))   # Filter out high probability clouds
        )
        return image.updateMask(mask)

    # Filter Sentinel-2 collection by space, time, and quality
    s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(region)
        .filterDate('2021-01-01', '2021-12-31')
        .map(mask_s2_clouds)
        .select(bands + ['SCL'])
    )

    # ---- 5. Create 10-Day Composites (Time Series) ----
    def get_10day_composites(collection):
        """Groups images into 10-day windows and takes the median to reduce noise."""
        start = ee.Date('2021-01-01')

        def make_image(i):
            i = ee.Number(i)
            start_date = start.advance(i.multiply(10), 'day')
            end_date = start_date.advance(10, 'day')
            
            subset = collection.filterDate(start_date, end_date)
            
            # Use median to handle remaining cloud artifacts; unmask with filler value
            median_img = subset.median().unmask(-9999).set({
                'system:time_start': start_date.millis(),
                'start_date': start_date.format('YYYY-MM-dd')
            })
            
            return median_img

        # Create 36 images (360 days total)
        return ee.ImageCollection(
            ee.List.sequence(0, 35).map(make_image)
        )
    
    s2_10day = get_10day_composites(s2)

    # ---- 6. Extract Band Values at Sample Points ----
    def sample_image(img):
        """Extracts the pixel values for all points from a single 10-day composite."""
        return img.sampleRegions(
            collection=all_samples,
            scale=10,
            geometries=True
        ).map(lambda f: f.set('time', img.get('start_date')))

    # Map the sampling function over the entire 10-day time series and flatten to a single table
    dataset = s2_10day.map(sample_image).flatten()

    # ---- 7. Export Result ----
    # Send the final dataset to Google Drive
    task = ee.batch.Export.table.toDrive(
        collection=dataset,
        description=filename,
        fileNamePrefix=filename,
        folder='EE_exports',
        fileFormat='CSV',
    )
    task.start()

## arkansas

In [ ]:

ar1=[-91.3, 35.8, -90.9, 36.1]
ar2=[-91.5, 34.5, -90.5, 35.5]
arkansas_regions = [
    ee.Geometry.Rectangle(ar1),
    ee.Geometry.Rectangle(ar2)
]

ar_region = ee.FeatureCollection(arkansas_regions).geometry()
arkCrops = [
    (5, 4677*13),
    (3, 2423*13),
    (1, 1522*14),
    (2, 762*30)
]
get_bands(ar_region, arkCrops, 'arkansas_batch1')

In [ ]:
arkansas_regions_batch2= [
    ee.Geometry.Rectangle( [-90.4, 35.9, -89.9, 36.4]),
    ee.Geometry.Rectangle([-90.4, 35.2, -89.9, 35.7]),
    ee.Geometry.Rectangle([-91.2, 33.8, -90.6, 34.4]),
    ee.Geometry.Rectangle([-91.0, 35.6, -90.6, 35.9]),
]
arkcrop_batch2=[(1, 698*10), (2, 1747*10)]

get_bands(ee.FeatureCollection(arkansas_regions_batch2).geometry(), arkcrop_batch2, 'arkansas_batch2')


--- Crop Distribution (only requested crops) ---
Crop 1: 634459.5921568628 points
Crop 2: 1056718.184313723 points

Total crop points: 11003974.458823523
Unspecified crop points: 9312796.682352938


## california

In [ ]:
cal1=[-120.0, 36.6, -119.7, 36.9]
cal2=[-122.5, 38.5, -121.0, 39.8]

calCrops = [
    (69, 2054),
    (3, 2037),
    (36, 974),
    (75, 783),
    (204, 640)
]

### grapes

In [16]:
california_regions = [
    ee.Geometry.Rectangle([-115.8, 32.8, -115.2, 33.4]) ,
    ee.Geometry.Rectangle([-120.9, 35.2, -120.3, 35.8]) ,
    ee.Geometry.Rectangle([-121.7, 37.6, -120.8, 38.5]) ,
    ee.Geometry.Rectangle([-120.8, 36.5, -119.8, 37.6]) ,
    ee.Geometry.Rectangle([-119.5, 34.9, -119, 36.5]) 
]

ca_region = ee.FeatureCollection(california_regions).union().geometry()
calCrops = [
    (69, 446867)
]
get_bands(ca_region,calCrops,'california_grapes')



--- Crop Distribution (only requested crops) ---
Crop 69: 2246867.019607843 points

Total crop points: 37849997.294117644
Unspecified crop points: 35603130.2745098


### rice

In [ ]:
california_regions = [
    ee.Geometry.Rectangle([-122.0, 38.5, -121.8, 39.8])
]

ca_region = ee.FeatureCollection(california_regions).union().geometry()
calCrops = [
    (3, 439881)
]
get_bands(ca_region,calCrops,'california_rice')




--- Crop Distribution (only requested crops) ---
Crop 3: 439881.0392156864 points

Total crop points: 2772329.7058823532
Unspecified crop points: 2332448.666666667


### almonds

In [ ]:
california_regions = [
    ee.Geometry.Rectangle([-119.5, 34.9, -119, 36.5]),
    ee.Geometry.Rectangle([-120.8, 36.5, -119.8, 37.6]),
    ee.Geometry.Rectangle([-121.7, 37.6, -120.8, 38.5]),
    ee.Geometry.Rectangle([-122.0, 38.5, -121.8, 39.8]),

]

ca_region = ee.FeatureCollection(california_regions).union().geometry()


calCrops = [
    (75, 5058328), 
]
get_bands(ca_region,calCrops,'california_almonds')



--- Crop Distribution (only requested crops) ---
Crop 75: 5058328.439215686 points

Total crop points: 32531267.756862745
Unspecified crop points: 27472939.31764706


### alfalfa

In [10]:
california_regions = [
    ee.Geometry.Rectangle([-120.8, 36.5, -119.8, 37.6]),
    ee.Geometry.Rectangle([-121.7, 37.6, -120.8, 38.5]),
    ee.Geometry.Rectangle([-122.7, 41.7, -121.1, 42.0]),
    ee.Geometry.Rectangle([-119.5, 34.9, -119, 36.5]),
    ee.Geometry.Rectangle([-122.0, 38.5, -121.8, 39.8]),
    ee.Geometry.Rectangle( [-119.5, 35.0, -118.8, 35.6]),
    ee.Geometry.Rectangle([-119.6, 35.8, -119.0, 36.3]),
    ee.Geometry.Rectangle([-120.9, 35.2, -120.3, 35.8]),
    ee.Geometry.Rectangle([-121.0, 39.8, -120.4, 40.4]),
]

ca_region = ee.FeatureCollection(california_regions).union().geometry()


calCrops = [
    (36, 454656),
]
get_bands(ca_region,calCrops,'california_alfafa')



--- Crop Distribution (only requested crops) ---
Crop 36: 1896163.3176470587 points

Total crop points: 47088531.25098039
Unspecified crop points: 45192367.93333333


### pistachios

In [ ]:
california_regions = [
    ee.Geometry.Rectangle([-119.6, 35.8, -119.0, 36.3]),

]
ca_region = ee.FeatureCollection(california_regions).union().geometry()


calCrops = [
    (204, 415790)
]
get_bands(ca_region,calCrops,'california_pistachios')


--- Crop Distribution (only requested crops) ---
Crop 36: 1896163.3176470587 points

Total crop points: 47088531.25098039
Unspecified crop points: 45192367.93333333

--- Crop Distribution (only requested crops) ---
Crop 204: 415790.0470588241 points

Total crop points: 3332704.6745098047
Unspecified crop points: 2916914.6274509807


## monitor tasks status

In [ ]:
tasks = ee.batch.Task.list()
for t in tasks:
        
        print(t.status().values())

In [ ]:
import ee
ee.Initialize()

ee.data.cancelOperation(
    'projects/1092489700890/operations/WTFLOVLAKDO2BNZG3HT46YTC'
)